# State-Dependent U.S. Equity Sector Rotation
## A Systematic Framework for Trend-Based Active Sector Allocation

# Block 3 — Weekly Signal Engine: TSMOM + SuperSmoother Oscillator

This block constructs the two signal families required for the core experiment:

1. **12-month time-series momentum (TSMOM)** for the conventional active-sector benchmark.
2. **Trend Following SuperSmoother — Accumulation Zones [JW]**, translated from the supplied Pine Script v6 source.

## Source-of-truth rule

The supplied Pine Script is the canonical specification for the proprietary indicator. The Python implementation below preserves its core numerical logic:

- Price SuperSmoother length = **5**
- Fast EMA = **20**
- Slow EMA = **50**
- Signal EMA = **25**
- Direction lookback = **2**
- BB volatility lookback = **20**
- BB-width EMA smoothing = **5**
- Upper / lower multipliers = **1.0 / 1.0**
- `oscillator = fastMA - slowMA`
- `spread = oscillator - signalLine`
- manual BBs centred on the signal line
- Pine-style crossover/crossunder event definitions

Supplied Pine source SHA-256:

`cddf115265809d9a29961e2eae64e6ad11008b3c63c48e2a8ae8a2939911cbd8`

Both signal families will ultimately be applied to the **same 52-week inverse-volatility strategic sector portfolio** created in Block 2.

Block 3 deliberately stops at numerical indicator values and primitive cross/direction events. The Pine **zone memory, priority rules, strategic position state, accumulation zones and profit-taking state machine** are translated separately in Block 4 so they can be tested independently.


In [ ]:
# ============================================================
# BLOCK 3.1 — ENVIRONMENT, DRIVE & MANIFESTS
# ============================================================

from pathlib import Path
from google.colab import drive
import json
import math
from collections import deque

drive.mount("/content/drive")

PROJECT_ROOT = Path("/content/drive/My Drive/Colab Notebooks/Sector Rotation Model/State-Dependent U.S. Equity Sector Rotation - A Systematic Framework for Trend-Based Active Sector Allocation")

DIRS = {
    "root": PROJECT_ROOT,
    "data_raw": PROJECT_ROOT / "data" / "raw",
    "data_processed": PROJECT_ROOT / "data" / "processed",
    "outputs": PROJECT_ROOT / "outputs",
    "figures": PROJECT_ROOT / "outputs" / "figures",
    "tables": PROJECT_ROOT / "outputs" / "tables",
    "manifests": PROJECT_ROOT / "manifests",
    "logs": PROJECT_ROOT / "logs",
}

BLOCK1_MANIFEST = DIRS["manifests"] / "block_1_research_configuration.json"
BLOCK2_MANIFEST = DIRS["manifests"] / "block_2_universe_data_weights.json"

for p in [BLOCK1_MANIFEST, BLOCK2_MANIFEST]:
    if not p.exists():
        raise FileNotFoundError(f"Required prior manifest not found: {p}")

with open(BLOCK1_MANIFEST, "r", encoding="utf-8") as f:
    block1 = json.load(f)

with open(BLOCK2_MANIFEST, "r", encoding="utf-8") as f:
    block2 = json.load(f)

CONFIG = block1["research_config"]

assert CONFIG["signal_frequency"] == "W-FRI"
assert CONFIG["rebalance_day"] == "MONDAY"
assert CONFIG["rebalance_execution_rule"] == "NEXT_US_TRADING_SESSION_AFTER_SIGNAL"

assert block2["strategic_weight_method"] == "INVERSE_VOLATILITY_52W"

print("Loaded Block 1 and Block 2 manifests.")
print("Canonical signal start:", block2["canonical_signal_start"])
print("Canonical execution start:", block2["canonical_execution_start"])
print("Strategic allocation inherited from Block 2:", block2["strategic_weight_method"])


Mounted at /content/drive
Loaded Block 1 and Block 2 manifests.
Canonical signal start: 2019-06-21
Canonical execution start: 2019-06-24
Strategic allocation inherited from Block 2: INVERSE_VOLATILITY_52W


In [ ]:
# ============================================================
# BLOCK 3.2 — IMPORTS & LOAD CANONICAL WEEKLY DATA
# ============================================================

import numpy as np
import pandas as pd
from datetime import datetime, timezone

WEEKLY_PRICE_FILE = (
    DIRS["data_processed"] / "weekly_signal_close_12_assets.parquet"
)
TIMING_FILE = (
    DIRS["data_processed"] / "weekly_signal_execution_calendar.parquet"
)

weekly_close = pd.read_parquet(WEEKLY_PRICE_FILE).sort_index()
weekly_timing = pd.read_parquet(TIMING_FILE).sort_index()

SECTOR_TICKERS = block2["sector_tickers"]
BENCHMARK_TICKER = block2["benchmark_ticker"]

assert all(t in weekly_close.columns for t in SECTOR_TICKERS)
assert BENCHMARK_TICKER in weekly_close.columns

print(
    f"Loaded {len(weekly_close):,} weekly signal observations "
    f"from {weekly_close.index.min().date()} to {weekly_close.index.max().date()}."
)


Loaded 428 weekly signal observations from 2018-06-22 to 2026-08-28.


## Exact indicator specification

The supplied Pine source defines:

```text
Price smoothing length  = 5
Fast EMA                = 20
Slow EMA                = 50
Signal EMA              = 25
Direction lookback      = 2
BB volatility lookback  = 20
BB width smoothing EMA  = 5
Upper multiplier        = 1.0
Lower multiplier        = 1.0
```

The Pine SuperSmoother recursion is:

\[
a_1=e^{-1.414\pi/L}
\]

\[
b_1=2a_1\cos(1.414\pi/L)
\]

\[
c_2=b_1,\qquad c_3=-a_1^2,\qquad c_1=1-c_2-c_3
\]

\[
SS_t=
c_1rac{P_t+P_{t-1}}{2}
+c_2SS_{t-1}
+c_3SS_{t-2}
\]

with Pine's `nz()` initialization represented by zeros.

The Bollinger width uses the **population** standard deviation (`ddof=0`), matching Pine's default `ta.stdev(..., biased=true)`.


In [ ]:
# ============================================================
# BLOCK 3.3 — SIGNAL PARAMETERS FROM SUPPLIED PINE SOURCE
# ============================================================

# Exact defaults from:
# "Trend Following SuperSmoother - Accumulation Zones [JW]"

PRICE_SMOOTHING_LENGTH = 5

FAST_MA_LENGTH = 20
SLOW_MA_LENGTH = 50
SIGNAL_LINE_LENGTH = 25

DIRECTION_LOOKBACK = 2

BB_VOLATILITY_LOOKBACK = 20
BB_WIDTH_SMOOTHING = 5
UPPER_BB_MULTIPLIER = 1.0
LOWER_BB_MULTIPLIER = 1.0

# Verify Block 1 has not drifted away from the Pine source.
assert int(CONFIG["fast_ema"]) == FAST_MA_LENGTH
assert int(CONFIG["slow_ema"]) == SLOW_MA_LENGTH
assert int(CONFIG["signal_ema"]) == SIGNAL_LINE_LENGTH
assert int(CONFIG["direction_lookback"]) == DIRECTION_LOOKBACK
assert int(CONFIG["bb_length"]) == BB_VOLATILITY_LOOKBACK
assert int(CONFIG["bb_smoothing"]) == BB_WIDTH_SMOOTHING
assert float(CONFIG["upper_multiplier"]) == UPPER_BB_MULTIPLIER
assert float(CONFIG["lower_multiplier"]) == LOWER_BB_MULTIPLIER

TSMOM_LOOKBACK_MONTHS = int(CONFIG["tsmom_lookback_months"])
TSMOM_LOOKBACK_WEEKS = round(TSMOM_LOOKBACK_MONTHS * 52 / 12)

signal_parameters = pd.Series(
    {
        "Price smoothing length": PRICE_SMOOTHING_LENGTH,
        "Fast EMA": FAST_MA_LENGTH,
        "Slow EMA": SLOW_MA_LENGTH,
        "Signal EMA": SIGNAL_LINE_LENGTH,
        "Direction lookback": DIRECTION_LOOKBACK,
        "BB volatility lookback": BB_VOLATILITY_LOOKBACK,
        "BB width smoothing": BB_WIDTH_SMOOTHING,
        "Upper BB multiplier": UPPER_BB_MULTIPLIER,
        "Lower BB multiplier": LOWER_BB_MULTIPLIER,
        "TSMOM lookback weeks": TSMOM_LOOKBACK_WEEKS,
    },
    name="Signal parameters",
).to_frame()

display(signal_parameters)


,Signal parameters
Price smoothing length,5.0
Fast EMA,20.0
Slow EMA,50.0
Signal EMA,25.0
Direction lookback,2.0
BB volatility lookback,20.0
BB width smoothing,5.0
Upper BB multiplier,1.0
Lower BB multiplier,1.0
TSMOM lookback weeks,52.0


In [ ]:
# ============================================================
# BLOCK 3.4 — 12-MONTH TSMOM SIGNAL
# ============================================================

sector_prices = weekly_close[SECTOR_TICKERS].copy()

tsmom_return = (
    sector_prices
    / sector_prices.shift(TSMOM_LOOKBACK_WEEKS)
    - 1.0
)

# Binary time-series momentum sign.
# +1 = positive 12m trend
# -1 = negative 12m trend
# NaN remains NaN during warm-up.
tsmom_sign = pd.DataFrame(
    np.where(
        tsmom_return.notna(),
        np.where(tsmom_return > 0.0, 1, -1),
        np.nan,
    ),
    index=tsmom_return.index,
    columns=tsmom_return.columns,
)

print("First complete TSMOM signal week:")
print(tsmom_sign.dropna(how="any").index.min())

display(tsmom_return.dropna(how="any").head())
display(tsmom_sign.dropna(how="any").head())


First complete TSMOM signal week:
2019-06-21 00:00:00


Ticker,XLC,XLY,XLP,XLE,XLF,XLV,XLI,XLK,XLB,XLRE,XLU
Date,,,,,,,,,,,
2019-06-21,-0.011645,0.088520,0.171028,-0.125795,0.026693,0.121856,0.084161,0.122193,0.006899,0.211582,0.241780
2019-06-28,0.003841,0.105669,0.160312,-0.132847,0.059818,0.128340,0.103842,0.139735,0.029306,0.165080,0.185817
2019-07-05,0.003000,0.118624,0.171205,-0.138222,0.078078,0.108006,0.097008,0.141502,0.029956,0.174400,0.179238
2019-07-12,0.001369,0.117303,0.168052,-0.126592,0.071746,0.075148,0.085544,0.134857,0.018077,0.181538,0.192000
2019-07-19,-0.015255,0.104330,0.167542,-0.133801,0.035113,0.075112,0.062585,0.128407,0.023843,0.173565,0.192957


Ticker,XLC,XLY,XLP,XLE,XLF,XLV,XLI,XLK,XLB,XLRE,XLU
Date,,,,,,,,,,,
2019-06-21,-1.0,1.0,1.0,-1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
2019-06-28,1.0,1.0,1.0,-1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
2019-07-05,1.0,1.0,1.0,-1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
2019-07-12,1.0,1.0,1.0,-1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
2019-07-19,-1.0,1.0,1.0,-1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0


In [ ]:
# ============================================================
# BLOCK 3.5 — PINE-PARITY RECURSIVE INDICATOR HELPERS
# ============================================================

class RecursiveEMA:
    """Recursive EMA equivalent to Pine ta.ema after initialization."""

    def __init__(self, length: int):
        self.length = int(length)
        self.alpha = 2.0 / (self.length + 1.0)
        self.value = None

    def update(self, x: float) -> float:
        x = float(x)
        if self.value is None:
            self.value = x
        else:
            self.value = (
                self.alpha * x
                + (1.0 - self.alpha) * self.value
            )
        return self.value


class SuperSmoother:
    """Two-pole SuperSmoother matching the supplied Pine recursion."""

    def __init__(self, length: int):
        self.length = int(length)

        a1 = math.exp(-1.414 * math.pi / self.length)
        b1 = 2.0 * a1 * math.cos(1.414 * math.pi / self.length)

        self.c2 = b1
        self.c3 = -a1 * a1
        self.c1 = 1.0 - self.c2 - self.c3

        self.prev_source = None
        self.prev_ss_1 = 0.0
        self.prev_ss_2 = 0.0

    def update(self, source: float) -> float:
        source = float(source)

        # Pine nz(source[1]) = 0 on the first observation.
        prev_source = (
            0.0
            if self.prev_source is None
            else self.prev_source
        )

        ss = (
            self.c1 * (source + prev_source) / 2.0
            + self.c2 * self.prev_ss_1
            + self.c3 * self.prev_ss_2
        )

        self.prev_source = source
        self.prev_ss_2 = self.prev_ss_1
        self.prev_ss_1 = ss

        return ss


In [ ]:
# ============================================================
# BLOCK 3.6 — SINGLE-SECTOR INDICATOR ENGINE
# ============================================================

def crossover(prev_a, a, prev_b, b) -> bool:
    if any(v is None or pd.isna(v) for v in [prev_a, a, prev_b, b]):
        return False
    return prev_a <= prev_b and a > b


def crossunder(prev_a, a, prev_b, b) -> bool:
    if any(v is None or pd.isna(v) for v in [prev_a, a, prev_b, b]):
        return False
    return prev_a >= prev_b and a < b


def compute_indicator_for_series(price: pd.Series) -> pd.DataFrame:
    """
    Compute the Pine-equivalent primitive signal series for one sector.

    This intentionally stops before the multi-week zone state machine.
    """

    ss = SuperSmoother(PRICE_SMOOTHING_LENGTH)
    fast_ema = RecursiveEMA(FAST_MA_LENGTH)
    slow_ema = RecursiveEMA(SLOW_MA_LENGTH)
    signal_ema = RecursiveEMA(SIGNAL_LINE_LENGTH)
    bb_width_ema = RecursiveEMA(BB_WIDTH_SMOOTHING)

    spread_window = deque(maxlen=BB_VOLATILITY_LOOKBACK)
    osc_history = deque(maxlen=DIRECTION_LOOKBACK + 1)
    signal_history = deque(maxlen=DIRECTION_LOOKBACK + 1)

    prev_osc = None
    prev_signal = None
    prev_upper = None
    prev_lower = None
    prev_osc_falling = False

    rows = []

    for date, close in price.items():
        if pd.isna(close):
            continue

        smoothed_price = ss.update(close)
        fast_ma = fast_ema.update(smoothed_price)
        slow_ma = slow_ema.update(smoothed_price)

        oscillator = fast_ma - slow_ma
        signal_line = signal_ema.update(oscillator)
        spread = oscillator - signal_line

        spread_window.append(spread)

        raw_spread_std = np.nan
        spread_std = np.nan
        upper_bb = np.nan
        lower_bb = np.nan

        if len(spread_window) >= BB_VOLATILITY_LOOKBACK:
            # Pine ta.stdev default is biased=true: population std.
            raw_spread_std = float(
                np.std(np.asarray(spread_window, dtype=float), ddof=0)
            )
            spread_std = bb_width_ema.update(raw_spread_std)

            upper_bb = (
                signal_line
                + UPPER_BB_MULTIPLIER * spread_std
            )
            lower_bb = (
                signal_line
                - LOWER_BB_MULTIPLIER * spread_std
            )

        osc_history.append(oscillator)
        signal_history.append(signal_line)

        oscillator_slope = np.nan
        signal_slope = np.nan

        if len(osc_history) > DIRECTION_LOOKBACK:
            oscillator_slope = (
                osc_history[-1]
                - osc_history[-1 - DIRECTION_LOOKBACK]
            )

        if len(signal_history) > DIRECTION_LOOKBACK:
            signal_slope = (
                signal_history[-1]
                - signal_history[-1 - DIRECTION_LOOKBACK]
            )

        osc_rising = (
            pd.notna(oscillator_slope)
            and oscillator_slope > 0
        )
        osc_falling = (
            pd.notna(oscillator_slope)
            and oscillator_slope < 0
        )

        signal_rising = (
            pd.notna(signal_slope)
            and signal_slope > 0
        )
        signal_falling = (
            pd.notna(signal_slope)
            and signal_slope < 0
        )

        # Exact Pine definitions:
        # oscFlat = not oscRising and not oscFalling
        # signalFlat = not signalRising and not signalFalling
        osc_flat = not bool(osc_rising) and not bool(osc_falling)
        signal_flat = not bool(signal_rising) and not bool(signal_falling)

        osc_turns_red = (
            bool(osc_falling)
            and not bool(prev_osc_falling)
        )

        # Pine cross events.
        cross_below_zero = (
            prev_osc is not None
            and prev_osc >= 0
            and oscillator < 0
        )

        signal_cross_above_zero = (
            prev_signal is not None
            and prev_signal <= 0
            and signal_line > 0
        )

        osc_cross_above_signal = crossover(
            prev_osc,
            oscillator,
            prev_signal,
            signal_line,
        )

        osc_cross_above_lower_bb = crossover(
            prev_osc,
            oscillator,
            prev_lower,
            lower_bb,
        )

        osc_cross_below_upper_bb = crossunder(
            prev_osc,
            oscillator,
            prev_upper,
            upper_bb,
        )

        ready = (
            pd.notna(lower_bb)
            and pd.notna(upper_bb)
            and pd.notna(oscillator_slope)
        )

        rows.append(
            {
                "date": date,
                "close": float(close),
                "smoothed_price": smoothed_price,
                "fast_ma": fast_ma,
                "slow_ma": slow_ma,
                "oscillator": oscillator,
                "signal_line": signal_line,
                "spread": spread,
                "raw_spread_std": raw_spread_std,
                "spread_std": spread_std,
                "upper_bb": upper_bb,
                "lower_bb": lower_bb,
                "oscillator_slope": oscillator_slope,
                "signal_slope": signal_slope,
                "osc_rising": bool(osc_rising),
                "osc_falling": bool(osc_falling),
                "signal_rising": bool(signal_rising),
                "signal_falling": bool(signal_falling),
                "osc_flat": bool(osc_flat),
                "signal_flat": bool(signal_flat),
                "osc_turns_red": bool(osc_turns_red),
                "cross_below_zero": bool(cross_below_zero),
                "signal_cross_above_zero": bool(signal_cross_above_zero),
                "osc_cross_above_signal": bool(osc_cross_above_signal),
                "osc_cross_above_lower_bb": bool(osc_cross_above_lower_bb),
                "osc_cross_below_upper_bb": bool(osc_cross_below_upper_bb),
                "indicator_ready": bool(ready),
            }
        )

        prev_osc = oscillator
        prev_signal = signal_line
        prev_upper = upper_bb if pd.notna(upper_bb) else None
        prev_lower = lower_bb if pd.notna(lower_bb) else None
        prev_osc_falling = bool(osc_falling)

    return (
        pd.DataFrame(rows)
        .set_index("date")
        .reindex(price.index)
    )


In [ ]:
# ============================================================
# BLOCK 3.7 — COMPUTE INDICATOR FOR ALL 11 SECTORS
# ============================================================

indicator_frames = {}

for ticker in SECTOR_TICKERS:
    indicator_frames[ticker] = compute_indicator_for_series(
        sector_prices[ticker]
    )

# Long-form research table: one row per signal week x sector.
indicator_long = (
    pd.concat(indicator_frames, names=["ticker", "date"])
    .swaplevel()
    .sort_index()
    .reset_index()
)

print(
    f"Indicator table: {len(indicator_long):,} "
    "week-sector observations."
)

display(indicator_long.head(15))


Indicator table: 4,708 week-sector observations.


,date,ticker,close,smoothed_price,fast_ma,slow_ma,oscillator,signal_line,spread,raw_spread_std,...,signal_falling,osc_flat,signal_flat,osc_turns_red,cross_below_zero,signal_cross_above_zero,osc_cross_above_signal,osc_cross_above_lower_bb,osc_cross_below_upper_bb,indicator_ready
0,2018-06-22,XLB,24.897751,8.096999,8.096999,8.096999,0.000000,0.000000,0.000000,NaN,...,False,True,True,False,False,False,False,False,False,False
1,2018-06-22,XLC,46.694256,15.185442,15.185442,15.185442,0.000000,0.000000,0.000000,NaN,...,False,True,True,False,False,False,False,False,False,False
2,2018-06-22,XLE,26.518709,8.624151,8.624151,8.624151,0.000000,0.000000,0.000000,NaN,...,False,True,True,False,False,False,False,False,False,False
3,2018-06-22,XLF,23.260498,7.564548,7.564548,7.564548,0.000000,0.000000,0.000000,NaN,...,False,True,True,False,False,False,False,False,False,False
4,2018-06-22,XLI,63.552570,20.667936,20.667936,20.667936,0.000000,0.000000,0.000000,NaN,...,False,True,True,False,False,False,False,False,False,False
5,2018-06-22,XLK,32.848316,10.682603,10.682603,10.682603,0.000000,0.000000,0.000000,NaN,...,False,True,True,False,False,False,False,False,False,False
6,2018-06-22,XLP,41.645058,13.543392,13.543392,13.543392,0.000000,0.000000,0.000000,NaN,...,False,True,True,False,False,False,False,False,False,False
7,2018-06-22,XLRE,24.642368,8.013946,8.013946,8.013946,0.000000,0.000000,0.000000,NaN,...,False,True,True,False,False,False,False,False,False,False
8,2018-06-22,XLU,19.779367,6.432450,6.432450,6.432450,0.000000,0.000000,0.000000,NaN,...,False,True,True,False,False,False,False,False,False,False
9,2018-06-22,XLV,74.078949,24.091220,24.091220,24.091220,0.000000,0.000000,0.000000,NaN,...,False,True,True,False,False,False,False,False,False,False


In [ ]:
# ============================================================
# BLOCK 3.8 — ADD TSMOM & EXECUTION DATES TO LONG-FORM SIGNAL TABLE
# ============================================================

# Give the wide TSMOM matrices explicit axis names before stacking.
# This guarantees that the resulting MultiIndex is named
# ["date", "ticker"], matching indicator_long exactly.
#
# future_stack=True uses pandas' current stack implementation and
# avoids the deprecation warning from the legacy stack behaviour.

tsmom_return_long = (
    tsmom_return
    .rename_axis(index="date", columns="ticker")
    .stack(future_stack=True)
    .rename("tsmom_12m_return")
)

tsmom_sign_long = (
    tsmom_sign
    .rename_axis(index="date", columns="ticker")
    .stack(future_stack=True)
    .rename("tsmom_sign")
)

indicator_indexed = (
    indicator_long
    .set_index(["date", "ticker"])
    .sort_index()
)

# Defensive contract checks before joining.
assert indicator_indexed.index.names == ["date", "ticker"]
assert tsmom_return_long.index.names == ["date", "ticker"]
assert tsmom_sign_long.index.names == ["date", "ticker"]

signal_long = (
    indicator_indexed
    .join(tsmom_return_long, how="left")
    .join(tsmom_sign_long, how="left")
    .reset_index()
)

timing_cols = weekly_timing[
    [
        "signal_observation_date",
        "execution_date",
        "execution_weekday",
    ]
].copy()

# The weekly timing table is indexed by the Friday-labelled signal week.
timing_cols.index = pd.to_datetime(timing_cols.index)
timing_cols.index.name = "date"

signal_long = signal_long.merge(
    timing_cols,
    left_on="date",
    right_index=True,
    how="left",
    validate="many_to_one",
)

signal_long = (
    signal_long
    .sort_values(["date", "ticker"])
    .reset_index(drop=True)
)

display(signal_long.head(15))


,date,ticker,close,smoothed_price,fast_ma,slow_ma,oscillator,signal_line,spread,raw_spread_std,...,signal_cross_above_zero,osc_cross_above_signal,osc_cross_above_lower_bb,osc_cross_below_upper_bb,indicator_ready,tsmom_12m_return,tsmom_sign,signal_observation_date,execution_date,execution_weekday
0,2018-06-22,XLB,24.897751,8.096999,8.096999,8.096999,0.000000,0.000000,0.000000,NaN,...,False,False,False,False,False,NaN,NaN,2018-06-22,2018-06-25,Monday
1,2018-06-22,XLC,46.694256,15.185442,15.185442,15.185442,0.000000,0.000000,0.000000,NaN,...,False,False,False,False,False,NaN,NaN,2018-06-22,2018-06-25,Monday
2,2018-06-22,XLE,26.518709,8.624151,8.624151,8.624151,0.000000,0.000000,0.000000,NaN,...,False,False,False,False,False,NaN,NaN,2018-06-22,2018-06-25,Monday
3,2018-06-22,XLF,23.260498,7.564548,7.564548,7.564548,0.000000,0.000000,0.000000,NaN,...,False,False,False,False,False,NaN,NaN,2018-06-22,2018-06-25,Monday
4,2018-06-22,XLI,63.552570,20.667936,20.667936,20.667936,0.000000,0.000000,0.000000,NaN,...,False,False,False,False,False,NaN,NaN,2018-06-22,2018-06-25,Monday
5,2018-06-22,XLK,32.848316,10.682603,10.682603,10.682603,0.000000,0.000000,0.000000,NaN,...,False,False,False,False,False,NaN,NaN,2018-06-22,2018-06-25,Monday
6,2018-06-22,XLP,41.645058,13.543392,13.543392,13.543392,0.000000,0.000000,0.000000,NaN,...,False,False,False,False,False,NaN,NaN,2018-06-22,2018-06-25,Monday
7,2018-06-22,XLRE,24.642368,8.013946,8.013946,8.013946,0.000000,0.000000,0.000000,NaN,...,False,False,False,False,False,NaN,NaN,2018-06-22,2018-06-25,Monday
8,2018-06-22,XLU,19.779367,6.432450,6.432450,6.432450,0.000000,0.000000,0.000000,NaN,...,False,False,False,False,False,NaN,NaN,2018-06-22,2018-06-25,Monday
9,2018-06-22,XLV,74.078949,24.091220,24.091220,24.091220,0.000000,0.000000,0.000000,NaN,...,False,False,False,False,False,NaN,NaN,2018-06-22,2018-06-25,Monday


In [ ]:
# ============================================================
# BLOCK 3.9 — SIGNAL-ENGINE DIAGNOSTICS
# ============================================================

ready_counts = (
    signal_long
    .groupby("ticker")["indicator_ready"]
    .agg(["sum", "count"])
)

ready_counts["ready_pct"] = (
    100.0 * ready_counts["sum"] / ready_counts["count"]
)

event_counts = (
    signal_long
    .groupby("ticker")[
        [
            "osc_turns_red",
            "cross_below_zero",
            "signal_cross_above_zero",
            "osc_cross_above_signal",
            "osc_cross_above_lower_bb",
            "osc_cross_below_upper_bb",
        ]
    ]
    .sum()
    .astype(int)
)

print("Indicator readiness:")
display(ready_counts)

print("\nPrimitive event counts:")
display(event_counts)

# Sanity checks
assert signal_long["oscillator"].notna().all()
assert signal_long["signal_line"].notna().all()

ready_rows = signal_long[signal_long["indicator_ready"]]
assert ready_rows["upper_bb"].notna().all()
assert ready_rows["lower_bb"].notna().all()
assert (ready_rows["upper_bb"] >= ready_rows["lower_bb"]).all()

print("\nSignal-engine sanity checks passed.")


Indicator readiness:


,sum,count,ready_pct
ticker,,,
XLB,409,428,95.560748
XLC,409,428,95.560748
XLE,409,428,95.560748
XLF,409,428,95.560748
XLI,409,428,95.560748
XLK,409,428,95.560748
XLP,409,428,95.560748
XLRE,409,428,95.560748
XLU,409,428,95.560748



Primitive event counts:


,osc_turns_red,cross_below_zero,signal_cross_above_zero,osc_cross_above_signal,osc_cross_above_lower_bb,osc_cross_below_upper_bb
ticker,,,,,,
XLB,23,4,3,8,4,8
XLC,15,1,2,6,4,5
XLE,22,2,3,8,6,6
XLF,20,4,3,9,6,6
XLI,18,2,3,6,5,10
XLK,20,2,2,10,6,10
XLP,22,2,2,10,8,8
XLRE,24,2,2,6,6,6
XLU,26,3,2,9,8,9



Signal-engine sanity checks passed.


In [ ]:
# ============================================================
# BLOCK 3.10 — CANONICAL SAMPLE DIAGNOSTIC
# ============================================================

canonical_start = pd.Timestamp(block2["canonical_signal_start"])
canonical_end = pd.Timestamp(block2["canonical_signal_end"])

canonical_signals = signal_long[
    (signal_long["date"] >= canonical_start)
    & (signal_long["date"] <= canonical_end)
].copy()

canonical_complete = (
    canonical_signals[
        [
            "oscillator",
            "signal_line",
            "upper_bb",
            "lower_bb",
            "tsmom_12m_return",
            "tsmom_sign",
            "execution_date",
        ]
    ]
    .notna()
    .all(axis=1)
)

print(
    "Canonical week-sector rows:",
    f"{len(canonical_signals):,}"
)
print(
    "Rows complete for both signal families and execution:",
    f"{canonical_complete.sum():,} / {len(canonical_complete):,}"
)

if not canonical_complete.all():
    incomplete = canonical_signals.loc[
        ~canonical_complete,
        ["date", "ticker"]
    ]
    display(incomplete.head(20))
else:
    print("Canonical sample is complete.")


Canonical week-sector rows: 4,125
Rows complete for both signal families and execution: 4,125 / 4,125
Canonical sample is complete.


In [ ]:
# ============================================================
# BLOCK 3.11 — SAVE SIGNAL DATASETS
# ============================================================

SIGNAL_LONG_PATH = (
    DIRS["data_processed"]
    / "weekly_sector_signal_engine_long.parquet"
)

TSMOM_RETURN_PATH = (
    DIRS["data_processed"]
    / "weekly_tsmom_12m_returns.parquet"
)

TSMOM_SIGN_PATH = (
    DIRS["data_processed"]
    / "weekly_tsmom_signs.parquet"
)

signal_long.to_parquet(
    SIGNAL_LONG_PATH,
    index=False,
)

tsmom_return.to_parquet(TSMOM_RETURN_PATH)
tsmom_sign.to_parquet(TSMOM_SIGN_PATH)

print("Saved:")
print(" ", SIGNAL_LONG_PATH)
print(" ", TSMOM_RETURN_PATH)
print(" ", TSMOM_SIGN_PATH)


Saved:
  /content/drive/My Drive/Colab Notebooks/Sector Rotation Model/State-Dependent U.S. Equity Sector Rotation - A Systematic Framework for Trend-Based Active Sector Allocation/data/processed/weekly_sector_signal_engine_long.parquet
  /content/drive/My Drive/Colab Notebooks/Sector Rotation Model/State-Dependent U.S. Equity Sector Rotation - A Systematic Framework for Trend-Based Active Sector Allocation/data/processed/weekly_tsmom_12m_returns.parquet
  /content/drive/My Drive/Colab Notebooks/Sector Rotation Model/State-Dependent U.S. Equity Sector Rotation - A Systematic Framework for Trend-Based Active Sector Allocation/data/processed/weekly_tsmom_signs.parquet


In [ ]:
# ============================================================
# BLOCK 3.12 — SAVE BLOCK 3 MANIFEST
# ============================================================

block3_manifest = {
    "project": block1["project"],
    "block": (
        "Block 3 - Weekly Signal Engine: "
        "TSMOM + SuperSmoother Oscillator"
    ),
    "created_utc": datetime.now(timezone.utc).isoformat(),

    "indicator_source": {
        "name": "Trend Following SuperSmoother - Accumulation Zones [JW]",
        "language": "Pine Script v6",
        "source_sha256": "cddf115265809d9a29961e2eae64e6ad11008b3c63c48e2a8ae8a2939911cbd8",
        "source_of_truth": True,
    },

    "signal_timing": {
        "signal_week_frequency": CONFIG["signal_frequency"],
        "configured_signal_observation": CONFIG["signal_observation"],
        "execution_rule": CONFIG["rebalance_execution_rule"],
        "rebalance_day": CONFIG["rebalance_day"],
    },

    "indicator_parameters": {
        "price_smoothing_length": PRICE_SMOOTHING_LENGTH,
        "fast_ma_length": FAST_MA_LENGTH,
        "slow_ma_length": SLOW_MA_LENGTH,
        "signal_line_length": SIGNAL_LINE_LENGTH,
        "direction_lookback": DIRECTION_LOOKBACK,
        "bb_volatility_lookback": BB_VOLATILITY_LOOKBACK,
        "bb_width_smoothing": BB_WIDTH_SMOOTHING,
        "upper_bb_multiplier": UPPER_BB_MULTIPLIER,
        "lower_bb_multiplier": LOWER_BB_MULTIPLIER,
        "bb_stdev_ddof": 0,
    },

    "tsmom": {
        "lookback_months": TSMOM_LOOKBACK_MONTHS,
        "lookback_weeks": TSMOM_LOOKBACK_WEEKS,
        "signal_definition": (
            "+1 if 12m return > 0; -1 otherwise"
        ),
    },

    "methodology": {
        "supersmoother": (
            "Two-pole recursive formula matching supplied Pine source"
        ),
        "ema": "Recursive EMA",
        "bollinger_width": (
            "Population standard deviation of oscillator-signal spread, "
            "then EMA-smoothed"
        ),
        "state_machine_included": False,
        "state_machine_next_block": True,
    },

    "canonical_signal_start": block2["canonical_signal_start"],
    "canonical_signal_end": block2["canonical_signal_end"],

    "saved_files": {
        "signal_engine_long": str(SIGNAL_LONG_PATH),
        "tsmom_returns": str(TSMOM_RETURN_PATH),
        "tsmom_signs": str(TSMOM_SIGN_PATH),
    },
}

BLOCK3_MANIFEST = (
    DIRS["manifests"] / "block_3_signal_engine.json"
)

with open(BLOCK3_MANIFEST, "w", encoding="utf-8") as f:
    json.dump(block3_manifest, f, indent=2)

print("Saved Block 3 manifest:")
print(BLOCK3_MANIFEST)


Saved Block 3 manifest:
/content/drive/My Drive/Colab Notebooks/Sector Rotation Model/State-Dependent U.S. Equity Sector Rotation - A Systematic Framework for Trend-Based Active Sector Allocation/manifests/block_3_signal_engine.json


## Pine parity boundary

Block 3 now mirrors the **numerical indicator layer** of the supplied Pine source.

The following Pine logic is intentionally **not** executed in Block 3 because it is stateful portfolio/zone logic and belongs in Block 4:

- `zoneMode`
- `earlyReversalExitMode`
- `earlyReversalSawBelowBB`
- `pullbackSawAboveBB`
- early-reversal entry/exit priority
- pullback entry/exit priority
- profit-taking entry/exit
- `positionHeld`
- strategic entry and zero-cross exit
- final accumulation / profit-taking state helpers

That separation is deliberate: first validate the oscillator and primitive events, then validate the state machine against those inputs.


In [ ]:
# ============================================================
# BLOCK 3.13 — FINAL STATUS
# ============================================================

first_indicator_ready = (
    signal_long.loc[
        signal_long["indicator_ready"],
        "date"
    ].min()
)

first_tsmom_ready = (
    tsmom_sign
    .dropna(how="any")
    .index.min()
)

summary = pd.Series(
    {
        "Sector count": len(SECTOR_TICKERS),
        "Signal frequency": CONFIG["signal_frequency"],
        "Execution rule": CONFIG["rebalance_execution_rule"],
        "Strategic allocation": block2["strategic_weight_method"],

        "Price smoothing length": PRICE_SMOOTHING_LENGTH,
        "Fast / Slow EMA": f"{FAST_MA_LENGTH} / {SLOW_MA_LENGTH}",
        "Signal EMA": SIGNAL_LINE_LENGTH,
        "BB lookback / smoothing": (
            f"{BB_VOLATILITY_LOOKBACK} / {BB_WIDTH_SMOOTHING}"
        ),
        "Direction lookback": DIRECTION_LOOKBACK,

        "TSMOM lookback weeks": TSMOM_LOOKBACK_WEEKS,
        "First complete TSMOM week": first_tsmom_ready.date(),
        "First indicator-ready week": first_indicator_ready.date(),

        "Canonical signal start": canonical_start.date(),
        "Canonical signal end": canonical_end.date(),

        "Canonical signal rows": len(canonical_signals),
        "Canonical complete rows": int(canonical_complete.sum()),
    },
    name="Block 3 status",
).to_frame()

display(summary)

print("\nBLOCK 3 COMPLETE")
print(
    "Next: Block 4 — State Machine & "
    "State-Dependent Active-Weight Engine"
)


,Block 3 status
Sector count,11
Signal frequency,W-FRI
Execution rule,NEXT_US_TRADING_SESSION_AFTER_SIGNAL
Strategic allocation,INVERSE_VOLATILITY_52W
Price smoothing length,5
Fast / Slow EMA,20 / 50
Signal EMA,25
BB lookback / smoothing,20 / 5
Direction lookback,2
TSMOM lookback weeks,52



BLOCK 3 COMPLETE
Next: Block 4 — State Machine & State-Dependent Active-Weight Engine
